# [핵심 솔루션] 메타데이터 역공학 로직

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher

In [ ]:
# ---------------------------------------------------------
# 1. 유사도 측정 함수 정의 (Intensity 역추적용)
# ---------------------------------------------------------
def calculate_change_ratio(row):
    # 원문과 교정문 사이의 유사도(0.0 ~ 1.0) 계산
    # ratio()가 1.0이면 완전 동일, 0.0이면 완전 다름
    s = SequenceMatcher(None, str(row['original']), str(row['corrected']))
    return s.ratio()

def map_intensity(ratio):
    # 유사도에 따라 Intensity 매핑 (임의 기준 설정)
    if ratio > 0.9:       # 90% 이상 동일 -> 아주 조금 고침
        return "weak"
    elif ratio > 0.6:     # 60~90% 동일 -> 적당히 고침
        return "moderate"
    else:                 # 60% 미만 동일 -> 대거 수정함
        return "strong"

# ---------------------------------------------------------
# 2. 데이터 로드 및 전처리
# ---------------------------------------------------------
# 예시: 사용자가 보여준 head 데이터를 df_new라고 가정
# 실제로는 pd.read_json("문장교정기록.json") 등으로 로드
df_new = pd.read_csv('./data/munch_labeled_RESUMED_200125.csv')
df_new.dropna(subset=['difficulty_label'], inplace=True)
# ---------------------------------------------------------
# 3. 결측 컬럼(Field, Intensity) 채우기
# ---------------------------------------------------------
print("⚙️ 메타데이터 자동 생성 중...")

# (1) Field: 정보가 없으므로 'none'로 통일 (안전한 선택)
df_new['field'] = 'none'

# (2) Intensity: 원문 vs 교정문 차이를 계산하여 자동 할당
# 유사도 계산
df_new['similarity'] = df_new.apply(calculate_change_ratio, axis=1)
# 강도 매핑
df_new['intensity'] = df_new['similarity'].apply(map_intensity)

# ---------------------------------------------------------
# 4. Feature Engineering (기존 모델 포맷 맞추기)
# ---------------------------------------------------------
# 컬럼명 통일 (original -> input_sentence)
df_new.rename(columns={'original': 'input_sentence'}, inplace=True)

# 기존 combine_features 함수 재사용
def combine_features(row):
    intensity_tag = f"[{str(row['intensity']).upper()}]"
    field_tag = f"[{str(row['field']).upper()}]"
    return f"{intensity_tag} {field_tag} {row['input_sentence']}"

df_new['text'] = df_new.apply(combine_features, axis=1)
df_new['label'] = df_new['difficulty_label'].astype(int)

# 결과 확인
print(df_new[['text', 'label', 'intensity', 'similarity']].head())

⚙️ 메타데이터 자동 생성 중...
                                                text  label intensity  \
0  [STRONG] [NONE] 폭발적으로 증가하는 그 정보를 어떻게 사람들한테 의미 ...      0    strong   
1  [STRONG] [NONE] 폭발적으로 증가하는 그 정보를 어떻게 사람들한테 의미 ...      1    strong   
2  [STRONG] [NONE] 폭발적으로 증가하는 그 정보를 어떻게 사람들한테 의미 ...      0    strong   
3  [STRONG] [NONE] 저는 이제 그 시각 디자인 전공에서 세부 분야로 이제 ...      0    strong   
4  [STRONG] [NONE] 저는 이제 그 시각 디자인 전공에서 세부 분야로 이제 ...      1    strong   

   similarity  
0    0.431138  
1    0.287500  
2    0.470588  
3    0.395161  
4    0.387097  


In [15]:
print(df_new.head())

                                      input_sentence  \
0  폭발적으로 증가하는 그 정보를 어떻게 사람들한테 의미 있게 만들어 줄 건가 이제 그...   
1  폭발적으로 증가하는 그 정보를 어떻게 사람들한테 의미 있게 만들어 줄 건가 이제 그...   
2  폭발적으로 증가하는 그 정보를 어떻게 사람들한테 의미 있게 만들어 줄 건가 이제 그...   
3  저는 이제 그 시각 디자인 전공에서 세부 분야로 이제 정보 디자인을 연구하고 주로 ...   
4  저는 이제 그 시각 디자인 전공에서 세부 분야로 이제 정보 디자인을 연구하고 주로 ...   

                                           corrected  \
0          폭발적으로 증가하는 정보를 어떻게 사람들에게 의미 있게 만들어 줄 것인가?   
1                 바로 이런 필요에 의해 정보 디자인이라는 영역이 생겨났습니다.   
2       정보 디자인은 사용자들이 잘 이용할 수 있는 맥락과 정황을 만들어주는 것입니다.   
3  저는 시각 디자인 전공에서 세부 분야로 정보 디자인을 연구하고 주로 프로젝트를 진행...   
4  그에 대한 이론적인 내용을 정리하고 책으로 출판하기도 하며 연구를 많이 했던 디자이...   

                                           reasoning  score  difficulty_label  \
0  The corrected sentence is a shortened version ...    3.0               0.0   
1  The corrected sentence is a significant simpli...    4.0               1.0   
2  The corrected sentence extracts a key definiti...    3.0               0.0   
3 

In [16]:
df_new.to_csv('./data/munch_labeled_RESUMED_200125_revised.csv',index=False, encoding='utf-8-sig')